In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import os
from PIL import Image
from tqdm import tqdm
from typing import Tuple, List
import argparse
import timm

In [30]:
class ImageDataset(Dataset):

    def __init__(self, root_dir: str, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        self.image_paths = []
        self.labels = []
        
        for class_name in self.classes:
            class_path = os.path.join(root_dir, class_name)
            for img_name in os.listdir(class_path):
                if img_name.lower().endswith('.png'):
                    self.image_paths.append(os.path.join(class_path, img_name))
                    self.labels.append(self.class_to_idx[class_name])
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx: int):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label


In [31]:
class CNN(nn.Module):
    
    def __init__(self, num_classes: int):
        super(CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x

In [32]:
def calculate_dataset_stats(dataset_path, device):
    # mean + std calculation for normalization
    dataset = ImageDataset(root_dir=dataset_path, transform=transforms.ToTensor())
    loader = DataLoader(dataset, batch_size=32)
    
    print("device!", device)
    mean = torch.zeros(3, device=device)
    std = torch.zeros(3, device=device)

    for images, _ in tqdm(loader):
        images = images.to(device)
        for i in range(3):
            # shape is (batch_size, 3, 224, 224) after ToTensor() in transform function
            mean[i] += images[:, i, :, :].mean() 
            std[i] += images[:, i, :, :].std()
    
    n_batches = len(loader)
    # average per batch
    mean = mean / n_batches
    std = std / n_batches
    
    return mean.cpu().tolist(), std.cpu().tolist()

In [33]:
def get_transforms(dataset_path, device):    
    # transform -- resize, augmentations, normalization
    
    mean, std = calculate_dataset_stats(dataset_path, device)

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])
    
    return train_transform, val_transform

In [34]:
def create_dataloaders(
    train_path,
    val_path,
    device,
    batch_size = 32,
    num_workers = 0,
):
    train_transform, val_transform = get_transforms(train_path, device)
    
    train_dataset = ImageDataset(root_dir=train_path, transform=train_transform)
    val_dataset = ImageDataset(root_dir=val_path, transform=val_transform)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    assert set(train_dataset.classes) == set(val_dataset.classes)
    
    return train_loader, val_loader, len(train_dataset.classes)


In [35]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs = 10,
    save_path = 'best_model.pth'
):
    best_val_acc = 0.0
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = running_loss / len(train_loader)
        train_acc = 100. * correct / total
        
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        
        val_loss = val_loss / len(val_loader)
        val_acc = 100. * correct / total
        
        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print('Model saved')
            
        print('-' * 50)

# Train CNN

In [ ]:
EPOCHS = 5
SAVE_PATH = 'best_model.pth'

dataset_path = os.path.join('data', 'spectrograms')

train_path = os.path.join(dataset_path, 'train')
val_path =  os.path.join(dataset_path, 'val')
train_loader, val_loader, num_classes = create_dataloaders(
        train_path,
        val_path,
        batch_size=32
    )
    
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = CNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs=EPOCHS,
    save_path=SAVE_PATH
)

# ViT (vit_tiny_patch16_224)

In [ ]:
MODEL_NAME = 'vit_tiny_patch16_224'

SAVE_PATH_VIT = 'best_model_vit.pth'

model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=num_classes
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs=EPOCHS,
    save_path=SAVE_PATH_VIT
)